# Add NCBI metadata

Load the filtered NCBI metadata TSV, format it to match the `ncbi_metadata_formatted` structure, and save to CSV.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
metadata_path = '/mnt/craig/pan_phylon/P_aeruginosa/data/interim/1a_ncbi_genome_metadata_filtered.tsv'
filtered_ncbi_metadata = pd.read_csv(metadata_path, sep='\t', dtype=object)
filtered_ncbi_metadata.head()

,Assembly Name,Assembly Accession,Assembly Paired Assembly Accession,Organism Name,Organism Taxonomic ID,ANI Check status,Organism Infraspecific Names Breed,Organism Infraspecific Names Strain,Organism Infraspecific Names Cultivar,Organism Infraspecific Names Ecotype,...,Type Material Display Text,CheckM marker set,CheckM completeness,CheckM contamination,contig_n50,total_sequence_length,gc_percent,checkm_completeness,checkm_contamination,cds
0,ASM676v1,GCA_000006765.1,GCF_000006765.1,Pseudomonas aeruginosa PAO1,208964,OK,NaN,PAO1,NaN,NaN,...,NaN,Pseudomonas aeruginosa,99.61,0.49,6264404,6264404,66.5,99.61,0.49,5571.0
1,ASM676v1,GCF_000006765.1,GCA_000006765.1,Pseudomonas aeruginosa PAO1,208964,OK,NaN,PAO1,NaN,NaN,...,NaN,Pseudomonas aeruginosa,99.61,0.49,6264404,6264404,66.5,99.61,0.49,5572.0
2,NCTC10332,GCA_001457615.1,GCF_001457615.1,Pseudomonas aeruginosa,287,OK,NaN,NCTC10332,NaN,NaN,...,assembly from type material,Pseudomonas aeruginosa,98.34,0.28,6316979,6316979,66.5,98.34,0.28,5760.0
3,NCTC10332,GCF_001457615.1,GCA_001457615.1,Pseudomonas aeruginosa,287,OK,NaN,NCTC10332,NaN,NaN,...,assembly from type material,Pseudomonas aeruginosa,98.34,0.28,6316979,6316979,66.5,98.34,0.28,5660.0
4,ASM104568v1,GCA_001045685.1,GCF_001045685.1,Pseudomonas aeruginosa DSM 50071 = NBRC 12689,1123015,OK,NaN,DSM 50071,NaN,NaN,...,assembly from type material,Pseudomonas aeruginosa,99.82,0.49,6317050,6317050,66.5,99.82,0.49,5710.0


# Run the following to download .json1 files
```bash
python ./Download_SeqReports.py /mnt/craig/pan_phylon/P_aeruginosa/data/interim/1a_ncbi_genome_metadata_filtered.tsv --output-dir /mnt/craig/pan_phylon/P_aeruginosa/data/raw/genomes/ncbi_seq_reports
```

In [4]:
seq_report_dir = Path('/mnt/craig/pan_phylon/P_aeruginosa/data/raw/genomes/ncbi_seq_reports')
seq_report_paths = sorted(seq_report_dir.glob('*.jsonl'))

records = []
for path in seq_report_paths:
    with path.open() as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))

print(f'Loaded {len(records)} seq-report records from {len(seq_report_paths)} files')

Loaded 2676 seq-report records from 2676 files


In [5]:
def attr_value(attributes, name):
    for attr in attributes or []:
        if attr.get('name') == name:
            return attr.get('value')
    return None

def join_values(values):
    cleaned = [str(v) for v in values or [] if v not in (None, '')]
    return ';'.join(cleaned) if cleaned else None

def flatten_bioproject_lineage(lineage):
    accessions = []
    titles = []
    for entry in lineage or []:
        for proj in entry.get('bioprojects', []) or []:
            acc = proj.get('accession')
            title = proj.get('title')
            if acc:
                accessions.append(acc)
            if title:
                titles.append(title)
    return {
        'bioproject_lineage_accessions': ';'.join(accessions) if accessions else None,
        'bioproject_lineage_titles': ';'.join(titles) if titles else None,
    }

def snake_case(value):
    out = []
    prev_underscore = False
    for ch in value:
        if ch.isalnum():
            out.append(ch.lower())
            prev_underscore = False
        else:
            if not prev_underscore:
                out.append('_')
                prev_underscore = True
    return ''.join(out).strip('_')

def normalize_record(rec):
    assembly_info = rec.get('assemblyInfo', {}) or {}
    biosample = assembly_info.get('biosample', {}) or {}
    attributes = biosample.get('attributes', []) or []
    organism = rec.get('organism', {}) or {}
    infraspecific = organism.get('infraspecificNames', {}) or {}
    stats = rec.get('assemblyStats', {}) or {}
    annotation = rec.get('annotationInfo', {}) or {}
    gene_counts = (annotation.get('stats', {}) or {}).get('geneCounts', {}) or {}
    ani = rec.get('averageNucleotideIdentity', {}) or {}
    checkm = rec.get('checkmInfo', {}) or {}

    strain = (
        infraspecific.get('strain')
        or biosample.get('strain')
        or attr_value(attributes, 'strain')
    )

    # organism_name = organism.get('organismName')
    # genome_name = (
    #     f"{organism_name} {strain}"
    #     if organism_name and strain
    #     else organism_name
    # )
    
############# modifying to consider cases where org name contains strain name already to avoid duplications
    organism_name = organism.get('organismName')
    if organism_name:
        if strain and strain not in organism_name:
            genome_name = f"{organism_name} {strain}"
        else:
            genome_name = organism_name
    else:
        genome_name = strain or None

    
    geo_loc = biosample.get('geoLocName') or attr_value(attributes, 'geo_loc_name')
    country = geo_loc.split(':')[0] if isinstance(geo_loc, str) else None

    accession = rec.get('accession') or rec.get('currentAccession')
    source_db = rec.get('sourceDatabase')
    paired = rec.get('pairedAccession')
    if source_db == 'SOURCE_DATABASE_REFSEQ':
        refseq_accession = accession
        genbank_accession = paired
    else:
        genbank_accession = accession
        refseq_accession = paired

    chromosomes = stats.get('totalNumberOfChromosomes')
    plasmids = chromosomes - 1 if isinstance(chromosomes, (int, float)) else None

    type_material = rec.get('typeMaterial', {}) or {}
    lineage = flatten_bioproject_lineage(assembly_info.get('bioprojectLineage'))

    biosample_ids = join_values(biosample.get('sampleIds'))
    biosample_models = join_values(biosample.get('models'))
    biosample_bioprojects = join_values(
        [bp.get('accession') for bp in biosample.get('bioprojects', []) or []]
    )

    return {
        'genome_id': accession,
        'genome_name': genome_name,
        'organism_name': organism_name,
        'taxon_id': organism.get('taxId'),
        'genome_status': assembly_info.get('assemblyLevel'),
        'strain': strain,
        'culture_collection': attr_value(attributes, 'culture_collection'),
        'type_strain': type_material.get('typeDisplayText'),
        'completion_date': assembly_info.get('releaseDate'),
        'bioproject_accession': assembly_info.get('bioprojectAccession'),
        'biosample_accession': biosample.get('accession'),
        'assembly_accession': accession,
        'genbank_accessions': genbank_accession,
        'refseq_accessions': refseq_accession,
        'sequencing_centers': assembly_info.get('submitter'),
        'sequencing_status': ani.get('taxonomyCheckStatus') or ani.get('matchStatus'),
        'sequencing_platform': assembly_info.get('sequencingTech'),
        'assembly_method': assembly_info.get('assemblyMethod'),
        'chromosomes': chromosomes,
        'plasmids': plasmids,
        'contigs': stats.get('numberOfContigs'),
        'sequences': stats.get('numberOfScaffolds'),
        'genome_length': stats.get('totalSequenceLength'),
        'gc_content': stats.get('gcPercent'),
        'refseq_cds': gene_counts.get('proteinCoding'),
        'isolation_source': (
            biosample.get('isolationSource')
            or attr_value(attributes, 'isolation_source')
        ),
        'collection_date': (
            biosample.get('collectionDate')
            or attr_value(attributes, 'collection_date')
        ),
        'isolation_country': country,
        'geographic_location': geo_loc,
        'other_environmental': geo_loc,
        'host_name': biosample.get('host') or attr_value(attributes, 'host'),
        'host_health': (
            biosample.get('hostDisease')
            or attr_value(attributes, 'host_disease')
        ),
        'comments': attr_value(attributes, 'comment'),
        # Seq-report extras (snake_case)
        'assembly_name': assembly_info.get('assemblyName'),
        'assembly_type': assembly_info.get('assemblyType'),
        'assembly_status': assembly_info.get('assemblyStatus'),
        'refseq_category': assembly_info.get('refseqCategory'),
        'assembly_release_date': assembly_info.get('releaseDate'),
        'source_database': source_db,
        'current_accession': rec.get('currentAccession'),
        'paired_accession': paired,
        'paired_assembly': assembly_info.get('pairedAssembly'),
        'bioproject_lineage_accessions': lineage['bioproject_lineage_accessions'],
        'bioproject_lineage_titles': lineage['bioproject_lineage_titles'],
        'biosample_last_updated': biosample.get('lastUpdated'),
        'biosample_publication_date': biosample.get('publicationDate'),
        'biosample_submission_date': biosample.get('submissionDate'),
        'biosample_package': biosample.get('package'),
        'biosample_status': biosample.get('status'),
        'biosample_description': biosample.get('description'),
        'biosample_owner': biosample.get('owner'),
        'biosample_collected_by': biosample.get('collectedBy'),
        'biosample_lat_lon': biosample.get('latLon'),
        'biosample_isolation_source': biosample.get('isolationSource'),
        'biosample_geo_loc_name': biosample.get('geoLocName'),
        'biosample_sample_ids': biosample_ids,
        'biosample_models': biosample_models,
        'biosample_bioprojects': biosample_bioprojects,
        'biosample_attributes_json': json.dumps(attributes) if attributes else None,
        'annotation_name': annotation.get('name'),
        'annotation_provider': annotation.get('provider'),
        'annotation_release_date': annotation.get('releaseDate'),
        'annotation_gene_total': gene_counts.get('total'),
        'annotation_gene_non_coding': gene_counts.get('nonCoding'),
        'annotation_gene_other': gene_counts.get('other'),
        'checkm_marker_set': checkm.get('checkmMarkerSet'),
        'checkm_marker_set_rank': checkm.get('checkmMarkerSetRank'),
        'checkm_version': checkm.get('checkmVersion'),
        'checkm_completeness': checkm.get('completeness'),
        'checkm_contamination': checkm.get('contamination'),
        'checkm_completeness_percentile': checkm.get('completenessPercentile'),
        'checkm_species_tax_id': checkm.get('checkmSpeciesTaxId'),
        'ani_match_status': ani.get('matchStatus'),
        'ani_category': ani.get('category'),
        'ani_submitted_organism': ani.get('submittedOrganism'),
        'ani_submitted_species': ani.get('submittedSpecies'),
        'ani_comment': ani.get('comment'),
        'ani_submitted_match_assembly': (ani.get('submittedAniMatch', {}) or {}).get('assembly'),
        'ani_submitted_match_ani': (ani.get('submittedAniMatch', {}) or {}).get('ani'),
        'ani_submitted_match_organism': (ani.get('submittedAniMatch', {}) or {}).get('organismName'),
        'ani_best_match_assembly': (ani.get('bestAniMatch', {}) or {}).get('assembly'),
        'ani_best_match_ani': (ani.get('bestAniMatch', {}) or {}).get('ani'),
        'ani_best_match_organism': (ani.get('bestAniMatch', {}) or {}).get('organismName'),
        'assembly_stats_total_ungapped_length': stats.get('totalUngappedLength'),
        'assembly_stats_contig_n50': stats.get('contigN50'),
        'assembly_stats_contig_l50': stats.get('contigL50'),
        'assembly_stats_scaffold_n50': stats.get('scaffoldN50'),
        'assembly_stats_scaffold_l50': stats.get('scaffoldL50'),
        'assembly_stats_component_sequences': stats.get('numberOfComponentSequences'),
        'assembly_stats_gc_count': stats.get('gcCount'),
        'assembly_stats_genome_coverage': stats.get('genomeCoverage'),
        'assembly_stats_atgc_count': stats.get('atgcCount'),
    }

formatted_records = [normalize_record(rec) for rec in records]
ncbi_metadata_formatted = pd.DataFrame(formatted_records)

accessions = set(filtered_ncbi_metadata['Assembly Accession'].dropna())
ncbi_metadata_formatted = ncbi_metadata_formatted[
    ncbi_metadata_formatted['assembly_accession'].isin(accessions)
]

# Add unused columns from filtered_ncbi_metadata with snake_case names
used_cols = set(ncbi_metadata_formatted.columns)
extra_cols = {}
for col in filtered_ncbi_metadata.columns:
    col_snake = snake_case(col)
    if col_snake in used_cols:
        continue
    extra_cols[col] = col_snake

filtered_extras = filtered_ncbi_metadata.rename(columns=extra_cols)
filtered_extras = filtered_extras[list(extra_cols.values()) + ['Assembly Accession']]
filtered_extras = filtered_extras.rename(columns={'Assembly Accession': 'assembly_accession'})

ncbi_metadata_formatted = ncbi_metadata_formatted.merge(
    filtered_extras,
    on='assembly_accession',
    how='left'
)

# Drop columns with identical content to avoid overlaps
dup_groups = {}
for col in ncbi_metadata_formatted.columns:
    series = ncbi_metadata_formatted[col]
    key = tuple(series.fillna('<<NA>>').astype(str).tolist())
    dup_groups.setdefault(key, []).append(col)

cols_to_drop = []
for cols in dup_groups.values():
    if len(cols) > 1:
        cols_to_drop.extend(cols[1:])

if cols_to_drop:
    ncbi_metadata_formatted = ncbi_metadata_formatted.drop(columns=cols_to_drop)

# Normalize genome_status to match prior formatting (e.g., 'Complete genome')
ncbi_metadata_formatted['genome_status'] = ncbi_metadata_formatted['genome_status'].apply(
    lambda x: x.split()[0] if isinstance(x, str) else x
)

ncbi_metadata_formatted.head()


,genome_id,genome_name,organism_name,taxon_id,genome_status,strain,culture_collection,type_strain,completion_date,bioproject_accession,...,assembly_stats_gc_count,assembly_stats_genome_coverage,assembly_stats_atgc_count,organism_infraspecific_names_strain,organism_infraspecific_names_isolate,assembly_stats_total_number_of_chromosomes,wgs_project_accession,annotation_count_gene_total,annotation_count_gene_protein_coding,annotation_count_gene_pseudogene
0,GCA_000006765.1,Pseudomonas aeruginosa PAO1,Pseudomonas aeruginosa PAO1,208964,Complete,PAO1,None,None,2006-07-07,PRJNA331,...,4169320,None,6264404,PAO1,NaN,1.0,NaN,5682.0,5571.0,5.0
1,GCA_000014625.1,Pseudomonas aeruginosa UCBPP-PA14,Pseudomonas aeruginosa UCBPP-PA14,208963,Complete,UCBPP-PA14,None,None,2006-10-06,PRJNA386,...,4333951,None,6537637,UCBPP-PA14,NaN,1.0,NaN,5977.0,5892.0,NaN
2,GCA_000026645.1,Pseudomonas aeruginosa LESB58,Pseudomonas aeruginosa LESB58,557722,Complete,LESB58,None,None,2008-12-24,PRJEA31101,...,4376792,None,6601757,LESB58,NaN,1.0,NaN,6067.0,5931.0,34.0
3,GCA_000226155.1,Pseudomonas aeruginosa M18,Pseudomonas aeruginosa M18,941193,Complete,M18,None,None,2011-09-19,PRJNA61423,...,4208227,None,6327754,M18,NaN,1.0,NaN,5770.0,5684.0,6.0
4,GCA_000271365.1,Pseudomonas aeruginosa DK2,Pseudomonas aeruginosa DK2,1093787,Complete,DK2,None,None,2012-06-20,PRJNA73815,...,4243057,None,6402658,DK2,NaN,1.0,NaN,5960.0,5884.0,NaN


In [6]:
output_path = '/mnt/craig/pan_phylon/P_aeruginosa/data/interim/1b_ncbi_genome_metadata_filtered.csv'
ncbi_metadata_formatted.to_csv(output_path, index=False)
output_path

'/mnt/craig/pan_phylon/P_aeruginosa/data/interim/1b_ncbi_genome_metadata_filtered.csv'